## Libraries

In [1]:
import numpy as np
import pandas as pd

from statsmodels.tsa.seasonal import STL
from statsmodels.stats.diagnostic import acorr_ljungbox

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

# NEW: cuML GPU Random Forest
from cuml.ensemble import RandomForestRegressor as cuRFRegressor

from tqdm import tqdm

## Config

In [2]:
TIME_COL   = "Date"
TARGET_COL = "AveragePrice"
ENTITY_COL = "AreaCode"

TRAIN_START_DATE = pd.Timestamp("2007-04-01")
TRAIN_END_DATE   = pd.Timestamp("2022-03-31")
TEST_START_DATE  = pd.Timestamp("2022-04-01")   # all dates >= this are test

# <<< FILL THESE FROM CV RESULTS >>>

best_lag_set = [1, 2, 3, 4, 5, 6, 12, 24]   
best_params  = {
    "n_estimators": 500,
    "max_depth": None,
    "min_samples_split": 5,
    "min_samples_leaf": 1,
    "max_features": 0.5,
    "bootstrap": True,
}

# continuous features
continuous_cols = [
    "AvgNeighbourPrice_lag1",
    "local_I_lag1",
    "area_km2",
    "centroid_x",
    "centroid_y",
    "CoL_distance_km",
    "LA_FE",
    "sdlt_perc_threshold",
    "dwelling_stock_per_1000",
    "population",
    "ashe_weekly",
    "base_rate",
    "claimant_count_prop",
    "planning_decisions_per_1000",
    "planning_granted_prop",
    "rail_station_entry_exit",
    "GDP",
    "CPIH",
    "LA_embed_0",
    "LA_embed_1",
    "LA_embed_2",
    "LA_embed_3",
    "LA_embed_4"
]

categorical_cols = [
    "LMIQuadrantlag1_2.0",
    "LMIQuadrantlag1_3.0",
    "LMIQuadrantlag1_4.0",
    "Region_East of England",
    "Region_London",
    "Region_North East",
    "Region_North West",
    "Region_South East",
    "Region_South West",
    "Region_West Midlands",
    "Region_Yorkshire and The Humber",
]

## Metric functions

In [3]:
def mae(y, yhat):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return np.mean(np.abs(y - yhat))

def rmse(y, yhat):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return np.sqrt(np.mean((y - yhat) ** 2))

def smape(y, yhat, eps=1e-8):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return 100.0 * np.mean(2.0 * np.abs(yhat - y) / (np.abs(y) + np.abs(yhat) + eps))

def mase(y, yhat, y_train, m=12, eps=1e-8):
    """Global MASE using in-sample seasonal naive with period m on y_train."""
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    y_train = np.asarray(y_train)
    if len(y_train) <= m:
        return np.nan
    naive_diff = np.abs(y_train[m:] - y_train[:-m])
    scale = np.mean(naive_diff) + eps
    return np.mean(np.abs(y - yhat)) / scale

def directional_accuracy(df, entity_col, time_col, target_col, pred_col):
    """Fraction of times sign of month-on-month change is correct."""
    df = df.sort_values([entity_col, time_col]).copy()
    df["y_diff"] = df.groupby(entity_col)[target_col].diff()
    df["yhat_diff"] = df.groupby(entity_col)[pred_col].diff()
    mask = df["y_diff"].notna() & df["yhat_diff"].notna()
    same_dir = np.sign(df.loc[mask, "y_diff"]) == np.sign(df.loc[mask, "yhat_diff"])
    return same_dir.mean()

def growth_rate_error(df, entity_col, time_col, target_col, pred_col, m=12):
    """
    12-month growth rate error:
    g_t = (y_t - y_{t-m}) / y_{t-m}
    Returns MAE of growth-rate error.
    """
    df = df.sort_values([entity_col, time_col]).copy()
    df["y_lag_m"] = df.groupby(entity_col)[target_col].shift(m)
    df["yhat_lag_m"] = df.groupby(entity_col)[pred_col].shift(m)

    mask = df["y_lag_m"].notna() & df["yhat_lag_m"].notna() & (df["y_lag_m"] != 0)
    y_gr = (df.loc[mask, target_col] - df.loc[mask, "y_lag_m"]) / df.loc[mask, "y_lag_m"]
    yhat_gr = (df.loc[mask, pred_col] - df.loc[mask, "yhat_lag_m"]) / df.loc[mask, "yhat_lag_m"]

    return np.mean(np.abs(y_gr - yhat_gr))

def morans_i(
    residuals,
    xs,
    ys,
    k=5,
    eps=1e-8,
    symmetric=True,
    row_standardize=True,
    permutations=0,
    random_state=None,
):
    """
    Compute Moran's I for residuals using k-nearest neighbours
    with inverse-distance weights.
    """
    residuals = np.asarray(residuals, dtype=float)
    xs = np.asarray(xs, dtype=float)
    ys = np.asarray(ys, dtype=float)

    N = len(residuals)
    if not (len(xs) == len(ys) == N):
        raise ValueError("residuals, xs, ys must all have the same length")

    # Center residuals
    x_mean = residuals.mean()
    x_dev = residuals - x_mean

    # Build kNN graph
    coords = np.column_stack([xs, ys])
    nbrs = NearestNeighbors(n_neighbors=k + 1).fit(coords)
    distances, indices = nbrs.kneighbors(coords)

    # Weight matrix W (dense; for large N you might switch to sparse)
    W = np.zeros((N, N), dtype=float)
    for i in range(N):
        neigh_idx = indices[i, 1:]          # skip self at index 0
        w = 1.0 / (distances[i, 1:] + eps)  # inverse-distance weights
        W[i, neigh_idx] = w

    # Optional symmetrisation
    if symmetric:
        W = 0.5 * (W + W.T)

    # Optional row standardisation
    if row_standardize:
        row_sums = W.sum(axis=1, keepdims=True)
        W = np.where(row_sums > 0, W / (row_sums + eps), 0.0)

    S0 = W.sum()

    # Moran's I numerator and denominator (vectorised)
    num = (W * (x_dev[:, None] * x_dev[None, :])).sum()
    den = (x_dev ** 2).sum() + eps

    I_obs = (N / S0) * (num / den)

    result = {
        "I": I_obs,
        "S0": S0,
        "permutations": None,
        "z_score": None,
        "p_value": None,
    }

    # Optional permutation test
    if permutations > 0:
        if isinstance(random_state, np.random.Generator):
            rng = random_state
        else:
            rng = np.random.default_rng(random_state)

        perm_I = np.empty(permutations, dtype=float)
        for b in range(permutations):
            perm = rng.permutation(x_dev)
            num_b = (W * (perm[:, None] * perm[None, :])).sum()
            perm_I[b] = (N / S0) * (num_b / den)

        mean_perm = perm_I.mean()
        std_perm = perm_I.std(ddof=1) + eps
        z = (I_obs - mean_perm) / std_perm

        extreme = np.sum(np.abs(perm_I - mean_perm) >= np.abs(I_obs - mean_perm))
        p_val = (extreme + 1) / (permutations + 1)

        result.update(
            {
                "permutations": perm_I,
                "z_score": z,
                "p_value": p_val,
            }
        )

    return result

## Load data

In [4]:
df = pd.read_excel("../../data/full_data.xlsx", parse_dates=[TIME_COL])
df = df.query('Date < "2024-03-31"')
df = df.sort_values([ENTITY_COL, TIME_COL])


## Rolling STL feature builder

In [5]:
def add_rolling_stl_components(
    df,
    entity_col,
    time_col,
    target_col,
    period=12,
    min_history=24,
    window=120,        # set None for expanding, or e.g. 120 to match your 10y window
    robust=True,
    show_progress=True,
):
    """
    For each LA, compute STL components at time t using only y up to time t.
    We assign the *last* STL values from the fitted history to that time t.

    IMPORTANT:
    - This creates stl_trend/stl_seasonal/stl_resid for each row.
    - You should only use *lags* of these components (e.g., lag1/lag12/lag24)
      when predicting y_t, otherwise you'd leak y_t into its own features.
    """
    df = df.sort_values([entity_col, time_col]).copy()
    df["stl_trend"] = np.nan
    df["stl_seasonal"] = np.nan
    df["stl_resid"] = np.nan

    # Precompute total iterations for tqdm
    groups = list(df.groupby(entity_col))
    total_steps = sum(len(sub) for _, sub in groups)

    iterator = tqdm(
        groups,
        desc="Rolling STL per LA",
        total=len(groups),
        leave=True,
        disable=not show_progress,
    )

    for la, sub in iterator:
        sub = sub.sort_values(time_col)
        y = sub[target_col].astype(float).values
        n = len(sub)

        for t in range(n):
            start = 0 if window is None else max(0, t - window + 1)
            hist = y[start : t + 1]

            if len(hist) < min_history or np.isnan(hist).any():
                continue

            try:
                res = STL(hist, period=period, robust=robust).fit()

                idx = sub.index[t]
                df.loc[idx, "stl_trend"]    = res.trend[-1]
                df.loc[idx, "stl_seasonal"] = res.seasonal[-1]
                df.loc[idx, "stl_resid"]    = res.resid[-1]

            except Exception:
                continue

    return df

## Evaluation

In [6]:

df = add_rolling_stl_components(
    df,
    entity_col=ENTITY_COL,
    time_col=TIME_COL,
    target_col=TARGET_COL,
    period=12,
    min_history=24,
    window=120,   # <- recommend matching your CV training window
    robust=True,
)

# =========================================================
# LAGGED STL FEATURES (best_lag_set) ACROSS FULL PANEL
# =========================================================
required_lag_cols = []
for lag in best_lag_set:
    for comp in ["stl_trend", "stl_seasonal", "stl_resid"]:
        col = f"{comp}_lag{lag}"
        df[col] = df.groupby(ENTITY_COL)[comp].shift(lag)
        required_lag_cols.append(col)

# =========================================================
# TRAIN / TEST SPLIT (feature period starts April 2007)
# =========================================================
mask_train = (df[TIME_COL] >= TRAIN_START_DATE) & (df[TIME_COL] <= TRAIN_END_DATE)
mask_test  = (df[TIME_COL] >= TEST_START_DATE)

df_train = df.loc[mask_train].copy()
df_test  = df.loc[mask_test].copy()

# Need all lag columns present
df_train = df_train.dropna(subset=required_lag_cols)
df_test  = df_test.dropna(subset=required_lag_cols)

# =========================================================
# STANDARDISE CONTINUOUS + LAG FEATURES
# =========================================================
scale_cols = continuous_cols + required_lag_cols
scaler = StandardScaler()
df_train[scale_cols] = scaler.fit_transform(df_train[scale_cols])
df_test[scale_cols]  = scaler.transform(df_test[scale_cols])

# =========================================================
# BUILD MATRICES
# =========================================================
feature_cols = continuous_cols + categorical_cols + required_lag_cols

X_train = df_train[feature_cols].to_numpy(dtype=np.float32)
y_train = df_train[TARGET_COL].to_numpy(dtype=np.float32)

X_test  = df_test[feature_cols].to_numpy(dtype=np.float32)
y_test  = df_test[TARGET_COL].to_numpy(dtype=np.float32)


# =========================================================
# TRAIN FINAL RANDOM FOREST
# =========================================================
cuml_params = best_params.copy()
# if you ever want "no max depth" from CV, set max_depth = -1 for cuML
if cuml_params.get("max_depth") is None:
    # drop it so cuML uses its own default max_depth
    cuml_params.pop("max_depth")

rf = cuRFRegressor(
    **cuml_params,
    random_state=42,
    n_streams=8,        # parallel CUDA streams
    output_type="numpy" # return numpy arrays
)

rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)  # numpy array

df_test["y_pred"] = y_pred
df_test["resid"] = df_test[TARGET_COL] - df_test["y_pred"]


Rolling STL per LA: 100%|██████████| 294/294 [02:55<00:00,  1.67it/s]


## Evaluation

In [7]:
# =========================================================
# GLOBAL ACCURACY
# =========================================================
global_mae   = mae(y_test, y_pred)
global_rmse  = rmse(y_test, y_pred)
global_smape = smape(y_test, y_pred)
global_mase  = mase(y_test, y_pred, y_train, m=12)

print("=== Global accuracy (ARX Ridge) ===")
print(f"MAE   : {global_mae:,.3f}")
print(f"RMSE  : {global_rmse:,.3f}")
print(f"sMAPE : {global_smape:,.3f}%")
print(f"MASE  : {global_mase:,.3f}")

# =========================================================
# ACROSS-LA CONSISTENCY
# =========================================================
la_mae = df_test.groupby(ENTITY_COL).apply(lambda g: mae(g[TARGET_COL].values, g["y_pred"].values))
median_mae = float(la_mae.median())
p75_mae    = float(la_mae.quantile(0.75))

print("\n=== Across-LA consistency ===")
print(f"Median LA MAE       : {median_mae:,.3f}")
print(f"75th percentile MAE : {p75_mae:,.3f}")

# =========================================================
# SPATIO-TEMPORAL DIAGNOSTICS
# =========================================================
# Moran's I: mean residual per LA over test period
la_resid_mean = df_test.groupby(ENTITY_COL)["resid"].mean()

centroids = (
    df_test
    .dropna(subset=["centroid_x", "centroid_y"])
    .sort_values(TIME_COL)
    .groupby(ENTITY_COL)
    .tail(1)
    .set_index(ENTITY_COL)[["centroid_x", "centroid_y"]]
)

# align
common = la_resid_mean.index.intersection(centroids.index)
la_resid_mean = la_resid_mean.loc[common]
centroids = centroids.loc[common]

mask_valid = centroids[["centroid_x", "centroid_y"]].notna().all(axis=1)
centroids_valid = centroids.loc[mask_valid]
la_resid_mean_valid = la_resid_mean.loc[centroids_valid.index]

print(f"\nLAs used for Moran's I: {len(centroids_valid)} / {len(la_resid_mean)}")

if len(centroids_valid) <= 1:
    I_moran = {"I": np.nan, "z_score": np.nan, "p_value": np.nan}
else:
    k_eff = min(5, len(centroids_valid) - 1)
    I_moran = morans_i(
        residuals=la_resid_mean_valid.values,
        xs=centroids_valid["centroid_x"].values,
        ys=centroids_valid["centroid_y"].values,
        k=k_eff,
        permutations=999,
        random_state=42,
    )
    print("\n=== Spatio-temporal diagnostics ===")
    print(f"Moran's I (mean residuals across LAs): {I_moran['I']:.4f}")
    print(f"Moran's I z score: {I_moran['z_score']:.4f}")
    print(f"Moran's I p value: {I_moran['p_value']:.4f}")

# Ljung–Box on monthly mean residuals (aggregate across LAs)
monthly_resid = df_test.groupby(TIME_COL)["resid"].mean().sort_index()
lb_res = acorr_ljungbox(monthly_resid, lags=[12], return_df=True)
q_stat = float(lb_res["lb_stat"].iloc[0])
p_val  = float(lb_res["lb_pvalue"].iloc[0])
print(f"Ljung–Box Q(12): stat={q_stat:.3f}, p={p_val:.4f}")

# =========================================================
# OPTIONAL: DIRECTIONAL ACCURACY & GROWTH-RATE ERROR
# =========================================================
dir_acc = directional_accuracy(df_test, ENTITY_COL, TIME_COL, TARGET_COL, "y_pred")
gre_mae = growth_rate_error(df_test, ENTITY_COL, TIME_COL, TARGET_COL, "y_pred", m=12)

print("\n=== Direction & growth ===")
print(f"Directional accuracy (MoM sign)   : {dir_acc:.3f}")
print(f"Growth-rate error MAE (12-month)  : {gre_mae:.4f}")

=== Global accuracy (ARX Ridge) ===
MAE   : 7,215.125
RMSE  : 13,938.951
sMAPE : 1.974%
MASE  : 0.319

=== Across-LA consistency ===
Median LA MAE       : 5,257.094
75th percentile MAE : 7,267.895

LAs used for Moran's I: 294 / 294

=== Spatio-temporal diagnostics ===
Moran's I (mean residuals across LAs): -0.0863
Moran's I z score: -2.8209
Moran's I p value: 0.0310
Ljung–Box Q(12): stat=72.634, p=0.0000

=== Direction & growth ===
Directional accuracy (MoM sign)   : 0.572
Growth-rate error MAE (12-month)  : 0.0293


/tmp/ipykernel_37326/2926894198.py:18: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  la_mae = df_test.groupby(ENTITY_COL).apply(lambda g: mae(g[TARGET_COL].values, g["y_pred"].values))


## Result output

In [8]:

output_path = "../../results/rf_gpu_final_test_results.xlsx"

summary_df = pd.DataFrame([{
    "model_type": "RandomForest",
    "lag_set": str(best_lag_set),
    "params": str(best_params),
    "MAE": global_mae,
    "RMSE": global_rmse,
    "sMAPE": global_smape,
    "MASE": global_mase,
    "Median_LA_MAE": median_mae,
    "P75_LA_MAE": p75_mae,
    "Morans_I": float(I_moran["I"]) if isinstance(I_moran, dict) else np.nan,
    "Morans_I_z": float(I_moran["z_score"]) if isinstance(I_moran, dict) and I_moran.get("z_score") is not None else np.nan,
    "Morans_I_p": float(I_moran["p_value"]) if isinstance(I_moran, dict) and I_moran.get("p_value") is not None else np.nan,
    "LjungBox_Q12": q_stat,
    "LjungBox_p": p_val,
    "Directional_Accuracy": dir_acc,
    "GrowthRateError_MAE": gre_mae
}])

la_mae_df = la_mae.reset_index()
la_mae_df.columns = [ENTITY_COL, "LA_MAE"]

with pd.ExcelWriter(output_path, engine="xlsxwriter") as writer:
    summary_df.to_excel(writer, sheet_name="Global_Summary", index=False)
    la_mae_df.to_excel(writer, sheet_name="LA_MAE", index=False)
    df_test[[ENTITY_COL, TIME_COL, TARGET_COL, "y_pred", "resid"]].to_excel(
        writer, sheet_name="Test_Predictions", index=False
    )

print(f"\nResults saved to: {output_path}")


Results saved to: ../../results/rf_gpu_final_test_results.xlsx


In [ ]:
from sklearn.metrics import mean_absolute_error
import numpy as np
import pandas as pd

def permutation_importance_mae(estimator, X, y, feature_names, n_repeats=10, seed=42):
    rng = np.random.default_rng(seed)
    baseline = mean_absolute_error(y, estimator.predict(X))
    importances = np.zeros((n_repeats, X.shape[1]))

    Xp = X.copy() if isinstance(X, np.ndarray) else X.copy(deep=True)

    for r in range(n_repeats):
        for j in range(X.shape[1]):
            if isinstance(Xp, np.ndarray):
                saved = Xp[:, j].copy()
                Xp[:, j] = rng.permutation(Xp[:, j])
                score = mean_absolute_error(y, estimator.predict(Xp))
                Xp[:, j] = saved
            else:
                saved = Xp.iloc[:, j].copy()
                Xp.iloc[:, j] = rng.permutation(Xp.iloc[:, j].to_numpy())
                score = mean_absolute_error(y, estimator.predict(Xp))
                Xp.iloc[:, j] = saved

            importances[r, j] = score - baseline  # increase in MAE is worse

    mean_imp = importances.mean(axis=0)
    std_imp = importances.std(axis=0)

    out = pd.DataFrame({"perm_imp_mae": mean_imp, "std": std_imp}, index=feature_names).sort_values("perm_imp_mae", ascending=False)
    return out, baseline

pi_rf, base_mae_rf = permutation_importance_mae(rf, X_test, y_test, feature_names, n_repeats=10)
print(pi_rf.head(20))

In [ ]:
pi_rf.to_csv("../../results/rf_gpu_feature_permutation_importance.csv", index=False)